# Final

In [1]:
# basic setup and import
import os
import json
import tempfile
import numpy as np
import pandas as pd

import mlflow
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

# ----------------------------
# MLflow experiment
# ----------------------------
mlflow.set_tracking_uri("sqlite:///../mlflow_db/mlflow.db")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment("heart_disease_prediction")

/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Tracking URI: sqlite:///../mlflow_db/mlflow.db


2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/23 10:54:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/23 10:54:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/23 10:54:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='/Users/gabi/codes/kaggle/02_predicting_heart_disease/mlruns/2', creation_time=1771627850202, experiment_id='2', last_update_time=1771627850202, lifecycle_stage='active', name='heart_disease_prediction', tags={}>

In [2]:
# treat dataset
def treat_dataset(data):
    # rename column names for easier access
    data.columns = [col.lower().replace(' ', '_') for col in data.columns]

    # convert heart_disease to binary
    if 'heart_disease' in data.columns:
        data['heart_disease'] = data['heart_disease'].apply(lambda x: 0 if (x == "Absence" or x == 0) else 1)

    return data


In [3]:
# we will merge the train dataset with the original dataset

train_data_new = pd.read_csv("data/train.csv")
train_data_new = treat_dataset(train_data_new)

train_data_original = pd.read_csv("data/train_original.csv")
train_data_original = treat_dataset(train_data_original)

# combine the two datasets
train_data = pd.concat([train_data_new, train_data_original], ignore_index=True)

# check if there are any duplicates in the combined dataset
duplicates = train_data.duplicated().sum()
print(f"Number of duplicate rows in the combined dataset: {duplicates}")


Number of duplicate rows in the combined dataset: 0


In [4]:
# check the structure of the combined dataset
print(train_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630270 entries, 0 to 630269
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  float64
 1   age                      630270 non-null  int64  
 2   sex                      630270 non-null  int64  
 3   chest_pain_type          630270 non-null  int64  
 4   bp                       630270 non-null  int64  
 5   cholesterol              630270 non-null  int64  
 6   fbs_over_120             630270 non-null  int64  
 7   ekg_results              630270 non-null  int64  
 8   max_hr                   630270 non-null  int64  
 9   exercise_angina          630270 non-null  int64  
 10  st_depression            630270 non-null  float64
 11  slope_of_st              630270 non-null  int64  
 12  number_of_vessels_fluro  630270 non-null  int64  
 13  thallium                 630270 non-null  int64  
 14  hear

In [7]:
# export the combined dataset to a new CSV file
train_data.to_csv("data/train_combined.csv", index=False)


## CatBoost - Retrain on full dataset

- Cat OOF AUC: 0.955533
- 0.9554778464282366

In [9]:
from catboost import CatBoostClassifier, Pool

# ----------------------------
# 1) Features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type", "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features

# ----------------------------
# 2) Load + treat train + test
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")
train_data = treat_dataset(train_data)

test_data = pd.read_csv("data/test.csv")
test_data = treat_dataset(test_data)

X_full = train_data[features].copy()
y_full = train_data[target].astype(int).values

X_test = test_data[features].copy()

cat_idx = [X_full.columns.get_loc(c) for c in categorical_features]

# ----------------------------
# 3) Params
# ----------------------------
v4_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
    thread_count=-1,

    iterations=3197,
    learning_rate=0.04571886361,
    depth=3,
    l2_leaf_reg=9.993655205,
    min_data_in_leaf=70,
    rsm=0.7483072281,
    subsample=0.9264181852,
    bootstrap_type="Bernoulli",
    border_count=192,
    one_hot_max_size=8,
    random_strength=0.8838868141,

    boosting_type="Plain",
    grow_policy="SymmetricTree",
)

# ----------------------------
# 4) CV OOF: print AUC per fold (seed ensemble inside fold)
# ----------------------------
N_SPLITS = 5
CV_SEED = 42
seeds = [0, 1, 2, 3, 4]  # your seed ensemble

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)

oof = np.zeros(len(X_full), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
    X_tr, y_tr = X_full.iloc[tr_idx], y_full[tr_idx]
    X_va, y_va = X_full.iloc[va_idx], y_full[va_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    va_pool = Pool(X_va, y_va, cat_features=cat_idx)

    fold_preds = []
    for s in seeds:
        model = CatBoostClassifier(**v4_params, random_seed=s)
        model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        fold_preds.append(model.predict_proba(va_pool)[:, 1])

    p_va = np.mean(np.vstack(fold_preds), axis=0)
    oof[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y_full, oof)
print(f"OOF AUC (all folds): {oof_auc:.6f}")

# ----------------------------
# 5) Train on FULL train + predict test (same seed ensemble)
# ----------------------------
train_pool = Pool(X_full, y_full, cat_features=cat_idx)
test_pool = Pool(X_test, cat_features=cat_idx)

test_preds = []
for s in seeds:
    model = CatBoostClassifier(**v4_params, random_seed=s)
    model.fit(train_pool)
    test_preds.append(model.predict_proba(test_pool)[:, 1])

p_ens_v4 = np.mean(np.vstack(test_preds), axis=0)

submission = pd.DataFrame({"id": test_data["id"], "heart_disease": p_ens_v4})
submission.to_csv("submissions/submission_catboost_v4_seed_ens_full_data.csv", index=False)

print("Saved: submission_catboost_v4_seed_ens_full_data.csv")

Fold 1 AUC: 0.955832
Fold 2 AUC: 0.955930
Fold 3 AUC: 0.954499
Fold 4 AUC: 0.955522
Fold 5 AUC: 0.955886
OOF AUC (all folds): 0.955533
Saved: submission_catboost_v4_seed_ens_full_data.csv


In [20]:
# Save the OOF predictions for potential future use (e.g., stacking)
oof_df = pd.DataFrame({"id": train_data["id"], "pred_cat_v4": oof})
oof_df.to_csv("submissions/oof_cat_v4_4seed_full_data.csv", index=False)

In [15]:
len(train_check)


630000

In [19]:
# check AUC in the train set (just for sanity check, since we don't have the test labels)
train_preds = pd.read_csv("submissions/oof_cat_v4_4seed.csv")
train_preds = train_preds.sort_values("id").reset_index(drop=True)

train_check = treat_dataset(pd.read_csv("data/train.csv"))

train_auc = roc_auc_score(train_check[target], train_preds["pred_cat_v4"])

print("Train AUC (sanity check):", train_auc)

Train AUC (sanity check): 0.9554778464282366


## CatBoost - Target Encoding

OOF AUC (all folds): 0.955468

In [47]:
# ----------------------------
# 0) Config
# ----------------------------
TARGET = "heart_disease"
N_SPLITS = 5
CV_SEED = 42
TE_ALPHA = 0.8          # 0.8 * category_mean + 0.2 * global_mean
TE_SPLITS_INNER = 5     # to make TE for training rows without leakage
TE_SEED = 42

seeds = [0, 1, 2, 3, 4]

# ----------------------------
# 1) Features
# ----------------------------
numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type",
                        "ekg_results", "slope_of_st", "thallium"]

# engineered categorical cross
CROSS_COL = "thallium_x_chest_pain_type"

# columns to target-encode (your notebook list + cross)
te_cols = ["chest_pain_type", "thallium", "ekg_results", "slope_of_st", "number_of_vessels_fluro", CROSS_COL]
te_feature_names = [f"{c}_target_enc" for c in te_cols]

features = numeric_features + categorical_features + [CROSS_COL] + te_feature_names

# ----------------------------
# 2) Load + treat
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")
train_data = treat_dataset(train_data)

test_data = treat_dataset(test_data)  # uncomment if kaggle_treated() does NOT apply treat_dataset

# Create cross feature as integer-coded category (fast, no object strings)
# (Assumes chest_pain_type is small like 1..4; using *10 is safe here)
train_data[CROSS_COL] = train_data["thallium"].astype(int) * 10 + train_data["chest_pain_type"].astype(int)
test_data[CROSS_COL]  = test_data["thallium"].astype(int) * 10 + test_data["chest_pain_type"].astype(int)

X_full_base = train_data[numeric_features + categorical_features + [CROSS_COL]].copy()
y_full = train_data[TARGET].astype(int).values
train_ids = train_data["id"].values

X_test_base = test_data[numeric_features + categorical_features + [CROSS_COL]].copy()

# CatBoost categorical indices (include the cross col as categorical)
cat_features_all = categorical_features + [CROSS_COL]

# ----------------------------
# 3) CatBoost params (keep your V4 for now)
# ----------------------------
v4_params = dict(
    loss_function="Logloss",
    eval_metric="AUC",
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
    thread_count=-1,

    iterations=3197,
    learning_rate=0.04571886361,
    depth=3,
    l2_leaf_reg=9.993655205,
    min_data_in_leaf=70,
    rsm=0.7483072281,
    subsample=0.9264181852,
    bootstrap_type="Bernoulli",
    border_count=192,
    one_hot_max_size=8,
    random_strength=0.8838868141,

    boosting_type="Plain",
    grow_policy="SymmetricTree",
)

# ----------------------------
# 4) Target encoding helpers (leak-free)
# ----------------------------
def _compute_mapping(col_series, y, alpha):
    """Return smoothed mapping: alpha*mean_by_cat + (1-alpha)*global_mean"""
    global_mean = float(np.mean(y))
    means = pd.Series(y).groupby(col_series).mean()
    smoothed = alpha * means + (1 - alpha) * global_mean
    return smoothed, global_mean

def _te_oof_for_train(col_series, y, n_splits, seed, alpha):
    """
    OOF target encoding for training rows:
    each row gets encoding computed from folds that did not include it.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    te = np.zeros(len(col_series), dtype=float)

    for tr_idx, va_idx in skf.split(np.zeros(len(y)), y):
        mapping, global_mean = _compute_mapping(col_series.iloc[tr_idx], y[tr_idx], alpha)
        te[va_idx] = col_series.iloc[va_idx].map(mapping).fillna(global_mean).values

    return te

def _te_apply_from_train(col_train, y_train, col_apply, alpha):
    """Fit mapping on full training subset and apply to val/test."""
    mapping, global_mean = _compute_mapping(col_train, y_train, alpha)
    return col_apply.map(mapping).fillna(global_mean).values

# ----------------------------
# 5) CV OOF for CatBoost (TE computed inside each outer fold)
# ----------------------------
outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
oof = np.zeros(len(train_data), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(outer.split(X_full_base, y_full), 1):
    X_tr = X_full_base.iloc[tr_idx].copy()
    y_tr = y_full[tr_idx]
    X_va = X_full_base.iloc[va_idx].copy()
    y_va = y_full[va_idx]

    # add TE features
    for c in te_cols:
        # OOF TE for training part (inner CV)
        X_tr[f"{c}_target_enc"] = _te_oof_for_train(X_tr[c], y_tr, n_splits=TE_SPLITS_INNER, seed=TE_SEED, alpha=TE_ALPHA)
        # leakage-free TE for validation part (fit on full outer-train, apply to outer-val)
        X_va[f"{c}_target_enc"] = _te_apply_from_train(X_tr[c], y_tr, X_va[c], alpha=TE_ALPHA)

    # build pools
    X_tr = X_tr[features]
    X_va = X_va[features]

    cat_idx = [X_tr.columns.get_loc(c) for c in cat_features_all]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    va_pool = Pool(X_va, y_va, cat_features=cat_idx)

    # seed ensemble inside fold
    fold_preds = []
    for s in seeds:
        model = CatBoostClassifier(**v4_params, random_seed=s)
        model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
        fold_preds.append(model.predict_proba(va_pool)[:, 1])

    p_va = np.mean(np.vstack(fold_preds), axis=0)
    oof[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y_full, oof)
print(f"OOF AUC (all folds): {oof_auc:.6f}")

pd.DataFrame({"id": train_ids, "pred_cat_te_full_data": oof}).to_csv("oof_cat_te_full_data.csv", index=False)
print("Saved: oof_cat_te_full_data.csv")

# ----------------------------
# 6) Train on FULL train + predict Kaggle test (TE from full train)
# ----------------------------
X_train_full = X_full_base.copy()
X_test_full = X_test_base.copy()

for c in te_cols:
    # for final training, you can either:
    # (A) compute OOF TE for train (more robust) OR (B) use full-train mapping (faster but leaks into train features)
    # We'll use (A) OOF TE for train to avoid training on leaked encodings.
    X_train_full[f"{c}_target_enc"] = _te_oof_for_train(X_train_full[c], y_full, n_splits=TE_SPLITS_INNER, seed=TE_SEED, alpha=TE_ALPHA)
    X_test_full[f"{c}_target_enc"] = _te_apply_from_train(X_train_full[c], y_full, X_test_full[c], alpha=TE_ALPHA)

X_train_full = X_train_full[features]
X_test_full = X_test_full[features]

cat_idx_full = [X_train_full.columns.get_loc(c) for c in cat_features_all]

train_pool = Pool(X_train_full, y_full, cat_features=cat_idx_full)
test_pool = Pool(X_test_full, cat_features=cat_idx_full)

test_preds = []
for s in seeds:
    model = CatBoostClassifier(**v4_params, random_seed=s)
    model.fit(train_pool)
    test_preds.append(model.predict_proba(test_pool)[:, 1])

p_test = np.mean(np.vstack(test_preds), axis=0)

sub = pd.DataFrame({"id": test_data["id"], TARGET: p_test})
sub.to_csv("submission_catboost_te_full_data.csv", index=False)
print("Saved: submission_catboost_te_full_data.csv")

Fold 1 AUC: 0.955750
Fold 2 AUC: 0.955868
Fold 3 AUC: 0.954442
Fold 4 AUC: 0.955478
Fold 5 AUC: 0.955829
OOF AUC (all folds): 0.955468
Saved: oof_cat_te_full_data.csv
Saved: submission_catboost_te_full_data.csv


## XGBoost - Retrain on full dataset

- XGB OOF AUC: 0.95 551 9
- original: 0.9553810007540939

In [21]:
from xgboost import XGBClassifier

# ----------------------------
# 1) Features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type", "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features

# ----------------------------
# 2) Params: xgb_reg_shallow
# ----------------------------
base_params = dict(
    n_estimators=8000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=8,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=1.0,
    reg_lambda=6.0,
    gamma=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    n_jobs=-1,
)

seeds = [42, 202, 999, 1337]

# ----------------------------
# 3) Prepare train/test (expects train_data + test_data already treated)
# ----------------------------
train_df = train_data.drop(columns=["id"]).copy()
X_full = train_df[features].copy()
y_full = train_df[target].astype(int).values
train_ids = train_data["id"].values

test_ids = test_data["id"].values
X_test = test_data.drop(columns=["id"], errors="ignore")[features].copy()

# cast categoricals
for c in categorical_features:
    X_full[c] = X_full[c].astype("category")
    X_test[c] = X_test[c].astype("category")

# ----------------------------
# 4) Stratified CV OOF (seed ensemble inside each fold)
# ----------------------------
N_SPLITS = 5
CV_SEED = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)

oof = np.zeros(len(X_full), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
    X_tr, y_tr = X_full.iloc[tr_idx], y_full[tr_idx]
    X_va, y_va = X_full.iloc[va_idx], y_full[va_idx]

    fold_preds = []
    for seed in seeds:
        params = dict(base_params)
        params["random_state"] = int(seed)

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)

        fold_preds.append(model.predict_proba(X_va)[:, 1])

    p_va = np.mean(np.vstack(fold_preds), axis=0)
    oof[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y_full, oof)
print(f"OOF AUC (all folds): {oof_auc:.6f}")

oof_df = pd.DataFrame({"id": train_ids, "pred_xgb_shallow_full_data": oof})
oof_df.to_csv("oof_xgb_shallow_full_data.csv", index=False)
print("Saved: oof_xgb_shallow_full_data.csv")

# ----------------------------
# 5) Train on FULL train + predict test (same seed ensemble)
# ----------------------------
test_preds = []
for seed in seeds:
    params = dict(base_params)
    params["random_state"] = int(seed)

    model = XGBClassifier(**params)
    model.fit(X_full, y_full)

    test_preds.append(model.predict_proba(X_test)[:, 1])

p_test_ens = np.mean(np.vstack(test_preds), axis=0)

sub = pd.DataFrame({"id": test_ids, target: p_test_ens})
sub.to_csv("submission_xgb_shallow_4seed_full_data.csv", index=False)
print("Saved: submission_xgb_shallow_4seed_full_data.csv")

Fold 1 AUC: 0.955853
Fold 2 AUC: 0.955909
Fold 3 AUC: 0.954446
Fold 4 AUC: 0.955538
Fold 5 AUC: 0.955862
OOF AUC (all folds): 0.955519
Saved: oof_xgb_shallow_full_data.csv
Saved: submission_xgb_shallow_4seed_full_data.csv


In [22]:
# check AUC on the train set (sanity check)
train_preds = pd.read_csv("submissions/oof_xgb_shallow_4seed.csv")
train_preds = train_preds.sort_values("id").reset_index(drop=True)

train_auc = roc_auc_score(train_check[target], train_preds["pred_xgb_shallow"])

print("Train AUC (sanity check):", train_auc)

Train AUC (sanity check): 0.9553810007540939


## XGBoost- Target Encoding

0.955483

In [49]:
from xgboost import XGBClassifier

# ----------------------------
# 0) Config
# ----------------------------
TARGET = "heart_disease"

N_SPLITS = 5
CV_SEED = 42

# Target encoding settings (match the notebook logic)
TE_ALPHA = 0.8          # 0.8 * category_mean + 0.2 * global_mean
TE_SPLITS_INNER = 5
TE_SEED = 42

# XGB seeds ensemble
seeds = [42, 202, 999, 1337]

# ----------------------------
# 1) Features
# ----------------------------
numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type",
                        "ekg_results", "slope_of_st", "thallium"]

# engineered categorical cross
CROSS_COL = "thallium_x_chest_pain_type"

# columns to target-encode (notebook list + cross)
te_cols = ["chest_pain_type", "thallium", "ekg_results", "slope_of_st", "number_of_vessels_fluro", CROSS_COL]
te_feature_names = [f"{c}_target_enc" for c in te_cols]

# model input columns
base_cols = numeric_features + categorical_features + [CROSS_COL]
model_features = base_cols + te_feature_names

# ----------------------------
# 2) Params: xgb_reg_shallow
# ----------------------------
base_params = dict(
    n_estimators=8000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=8,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=1.0,
    reg_lambda=6.0,
    gamma=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    n_jobs=-1,
)

# ----------------------------
# 3) Load + treat
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")
train_data = treat_dataset(train_data)


test_data = treat_dataset(test_data)

# engineered cross (integer-coded, fast)
train_data[CROSS_COL] = train_data["thallium"].astype(int) * 10 + train_data["chest_pain_type"].astype(int)
test_data[CROSS_COL]  = test_data["thallium"].astype(int) * 10 + test_data["chest_pain_type"].astype(int)

y_full = train_data[TARGET].astype(int).values
train_ids = train_data["id"].values

X_full_base = train_data[base_cols].copy()
X_test_base = test_data[base_cols].copy()

# cast categorical cols (including the cross) to pandas category for XGB
cat_cols_all = categorical_features + [CROSS_COL]
for c in cat_cols_all:
    X_full_base[c] = X_full_base[c].astype("category")
    X_test_base[c] = X_test_base[c].astype("category")

# ----------------------------
# 4) Target encoding helpers (leak-free)
# ----------------------------
def _compute_mapping(col_series, y, alpha):
    global_mean = float(np.mean(y))

    # IMPORTANT: use a stable, non-categorical key
    key = col_series.astype("int64") if str(col_series.dtype) == "category" else col_series
    means = pd.Series(y).groupby(key).mean()

    smoothed = alpha * means + (1 - alpha) * global_mean
    return smoothed, global_mean

def _te_oof_for_train(col_series, y, n_splits, seed, alpha):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    te = np.zeros(len(col_series), dtype=float)

    for tr_idx, va_idx in skf.split(np.zeros(len(y)), y):
        mapping, global_mean = _compute_mapping(col_series.iloc[tr_idx], y[tr_idx], alpha)

        # IMPORTANT: map on the same key type, then force float
        key_va = col_series.iloc[va_idx]
        if str(key_va.dtype) == "category":
            key_va = key_va.astype("int64")

        mapped = key_va.map(mapping)

        te[va_idx] = mapped.astype("float64").fillna(global_mean).values

    return te

def _te_apply_from_train(col_train, y_train, col_apply, alpha):
    mapping, global_mean = _compute_mapping(col_train, y_train, alpha)

    key_apply = col_apply
    if str(key_apply.dtype) == "category":
        key_apply = key_apply.astype("int64")

    mapped = key_apply.map(mapping)
    return mapped.astype("float64").fillna(global_mean).values

# ----------------------------
# 5) Outer CV: OOF for XGB (TE computed inside each outer fold)
# ----------------------------
outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
oof = np.zeros(len(train_data), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(outer.split(X_full_base, y_full), 1):
    X_tr = X_full_base.iloc[tr_idx].copy()
    y_tr = y_full[tr_idx]
    X_va = X_full_base.iloc[va_idx].copy()
    y_va = y_full[va_idx]

    # add TE features (leak-free)
    for c in te_cols:
        X_tr[f"{c}_target_enc"] = _te_oof_for_train(X_tr[c], y_tr, n_splits=TE_SPLITS_INNER, seed=TE_SEED, alpha=TE_ALPHA)
        X_va[f"{c}_target_enc"] = _te_apply_from_train(X_tr[c], y_tr, X_va[c], alpha=TE_ALPHA)

    X_tr = X_tr[model_features]
    X_va = X_va[model_features]

    fold_preds = []
    for seed in seeds:
        params = dict(base_params)
        params["random_state"] = int(seed)

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)

        fold_preds.append(model.predict_proba(X_va)[:, 1])

    p_va = np.mean(np.vstack(fold_preds), axis=0)
    oof[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y_full, oof)
print(f"OOF AUC (all folds): {oof_auc:.6f}")

pd.DataFrame({"id": train_ids, "pred_xgb_shallow_te_full_data": oof}).to_csv(
    "oof_xgb_shallow_te_full_data.csv", index=False
)
print("Saved: oof_xgb_shallow_te_full_data.csv")

# ----------------------------
# 6) Train on FULL train + predict Kaggle test
# ----------------------------
X_train_full = X_full_base.copy()
X_test_full = X_test_base.copy()

# TE for full-train features:
# - train TE as OOF (to avoid using its own label)
# - test TE from full-train mapping
for c in te_cols:
    X_train_full[f"{c}_target_enc"] = _te_oof_for_train(X_train_full[c], y_full, n_splits=TE_SPLITS_INNER, seed=TE_SEED, alpha=TE_ALPHA)
    X_test_full[f"{c}_target_enc"] = _te_apply_from_train(X_train_full[c], y_full, X_test_full[c], alpha=TE_ALPHA)

X_train_full = X_train_full[model_features]
X_test_full = X_test_full[model_features]

test_preds = []
for seed in seeds:
    params = dict(base_params)
    params["random_state"] = int(seed)

    model = XGBClassifier(**params)
    model.fit(X_train_full, y_full)

    test_preds.append(model.predict_proba(X_test_full)[:, 1])

p_test_ens = np.mean(np.vstack(test_preds), axis=0)

sub = pd.DataFrame({"id": test_data["id"], TARGET: p_test_ens})
sub.to_csv("submission_xgb_shallow_te_full_data.csv", index=False)
print("Saved: submission_xgb_shallow_te_full_data.csv")

Fold 1 AUC: 0.955836
Fold 2 AUC: 0.955856
Fold 3 AUC: 0.954431
Fold 4 AUC: 0.955515
Fold 5 AUC: 0.955848
OOF AUC (all folds): 0.955483
Saved: oof_xgb_shallow_te_full_data.csv
Saved: submission_xgb_shallow_te_full_data.csv


## RealMLP

RealMLP OOF AUC: 0.953321


In [50]:
from pytabkit import RealMLP_TD_Classifier

# ----------------------------
# 0) Config
# ----------------------------
BASE_SEED = 42
N_SPLITS = 5

# RealMLP training knobs (CPU)
N_EPOCHS = 64
BATCH_SIZE = 4096
HIDDEN_SIZES = [256, 256]
LR = 0.04

# Seed ensemble for RealMLP (set to [0] to disable ensembling)
SEED_OFFSETS = [0]          # e.g. [0, 1, 2] if you really want 3 seeds

# ----------------------------
# 1) Features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type",
                        "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features

# ----------------------------
# 2) Load + treat
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")  # use full data if that’s your new baseline
train_data = treat_dataset(train_data)


test_data = treat_dataset(test_data)  # uncomment if kaggle_treated() doesn't treat

X = train_data[features].copy()
y = train_data[target].astype(int).values
X_test = test_data[features].copy()

# ensure categoricals are integer-coded
for c in categorical_features:
    X[c] = X[c].astype(int)
    X_test[c] = X_test[c].astype(int)

train_ids = train_data["id"].values

# ----------------------------
# 3) CV OOF + test preds (same folds)
# ----------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED)

oof = np.zeros(len(X), dtype=float)
test_preds_all = []  # store fold-level preds for each seed offset

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    fold_va_preds = []
    fold_te_preds = []

    for off in SEED_OFFSETS:
        model = RealMLP_TD_Classifier(
            device="cpu",
            random_state=BASE_SEED + 1000 * off + fold,
            n_cv=1,
            n_refit=0,
            n_epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            hidden_sizes=HIDDEN_SIZES,
            lr=LR,
            verbosity=0,
            use_ls=False,
        )

        model.fit(X_tr, y_tr, X_va, y_va, cat_col_names=categorical_features)

        fold_va_preds.append(model.predict_proba(X_va)[:, 1])
        fold_te_preds.append(model.predict_proba(X_test)[:, 1])

    # average across RealMLP seeds for this fold
    p_va = np.mean(np.vstack(fold_va_preds), axis=0)
    oof[va_idx] = p_va

    # save averaged fold test preds
    p_te = np.mean(np.vstack(fold_te_preds), axis=0)
    test_preds_all.append(p_te)

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y, oof)
print(f"RealMLP OOF AUC: {oof_auc:.6f}")

pd.DataFrame({"id": train_ids, "pred_realmlp_full_data": oof}).to_csv("oof_realmlp_full_data.csv", index=False)
print("Saved: oof_realmlp_full_data.csv")

# average across folds for the final test prediction
p_test = np.mean(np.vstack(test_preds_all), axis=0)

sub = pd.DataFrame({"id": test_data["id"], target: p_test})
sub.to_csv("submission_realmlp_full_data.csv", index=False)
print("Saved: submission_realmlp_full_data.csv")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 1 AUC: 0.953703


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 2 AUC: 0.953894


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 3 AUC: 0.952444


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 4 AUC: 0.953385


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 5 AUC: 0.953619
RealMLP OOF AUC: 0.953321
Saved: oof_realmlp_full_data.csv
Saved: submission_realmlp_full_data.csv


## RealMLP with val_metric_name: 'val_metric_name': '1-auc_ovr'

In [ ]:
from pytabkit import RealMLP_TD_Classifier

# ----------------------------
# 0) Config
# ----------------------------
BASE_SEED = 42
N_SPLITS = 5

# RealMLP training knobs (CPU)
N_EPOCHS = 64
BATCH_SIZE = 4096
HIDDEN_SIZES = [256, 256]
LR = 0.04

# Seed ensemble for RealMLP (set to [0] to disable ensembling)
SEED_OFFSETS = [0]          # e.g. [0, 1, 2] if you really want 3 seeds

# ----------------------------
# 1) Features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type",
                        "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features

# ----------------------------
# 2) Load + treat
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")  # use full data if that’s your new baseline
train_data = treat_dataset(train_data)

test_data = pd.read_csv("data/test.csv")
test_data = treat_dataset(test_data)  # uncomment if kaggle_treated() doesn't treat

X = train_data[features].copy()
y = train_data[target].astype(int).values
X_test = test_data[features].copy()

# ensure categoricals are integer-coded
for c in categorical_features:
    X[c] = X[c].astype(int)
    X_test[c] = X_test[c].astype(int)

train_ids = train_data["id"].values

# ----------------------------
# 3) CV OOF + test preds (same folds)
# ----------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED)

oof = np.zeros(len(X), dtype=float)
test_preds_all = []  # store fold-level preds for each seed offset

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    fold_va_preds = []
    fold_te_preds = []

    for off in SEED_OFFSETS:
        model = RealMLP_TD_Classifier(
            device="cpu",
            random_state=BASE_SEED + 1000 * off + fold,
            n_cv=1,
            n_refit=0,
            n_epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            hidden_sizes=HIDDEN_SIZES,
            lr=LR,
            verbosity=0,
            use_ls=False,
            val_metric_name = "1-auc_ovr"
        )

        model.fit(X_tr, y_tr, X_va, y_va, cat_col_names=categorical_features)

        fold_va_preds.append(model.predict_proba(X_va)[:, 1])
        fold_te_preds.append(model.predict_proba(X_test)[:, 1])

    # average across RealMLP seeds for this fold
    p_va = np.mean(np.vstack(fold_va_preds), axis=0)
    oof[va_idx] = p_va

    # save averaged fold test preds
    p_te = np.mean(np.vstack(fold_te_preds), axis=0)
    test_preds_all.append(p_te)

    fold_auc = roc_auc_score(y_va, p_va)
    print(f"Fold {fold} AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y, oof)
print(f"RealMLP OOF AUC: {oof_auc:.6f}")

pd.DataFrame({"id": train_ids, "pred_realmlp_full_data": oof}).to_csv("oof_realmlp_full_data.csv", index=False)
print("Saved: oof_realmlp_full_data.csv")

# average across folds for the final test prediction
p_test = np.mean(np.vstack(test_preds_all), axis=0)

sub = pd.DataFrame({"id": test_data["id"], target: p_test})
sub.to_csv("submission_realmlp_full_data.csv", index=False)
print("Saved: submission_realmlp_full_data.csv")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 1 AUC: 0.953849


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=64` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstan

Fold 2 AUC: 0.953920


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/anaconda3/envs/ml_kaggle/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


## AdaBoost

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

# ----------------------------
# 1) Features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type",
                        "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features

# ----------------------------
# 2) Load + treat
# ----------------------------
train_data = pd.read_csv("data/train_combined.csv")
train_data = treat_dataset(train_data)

X_full = train_data[features].copy()
y_full = train_data[target].astype(int).values
train_ids = train_data["id"].values

# ----------------------------
# 3) AdaBoost params (start point)
# ----------------------------
base_tree = DecisionTreeClassifier(
    max_depth=2,
    min_samples_leaf=50,
    random_state=42
)

ada = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=800,
    learning_rate=0.05,
    algorithm="SAMME.R",
    random_state=42
)

# ----------------------------
# 4) Stratified CV OOF
# ----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X_full), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
    X_tr, y_tr = X_full.iloc[tr_idx], y_full[tr_idx]
    X_va, y_va = X_full.iloc[va_idx], y_full[va_idx]

    model = ada
    model.fit(X_tr, y_tr)
    p_va = model.predict_proba(X_va)[:, 1]
    oof[va_idx] = p_va

    print(f"Fold {fold} AUC: {roc_auc_score(y_va, p_va):.6f}")

print(f"OOF AUC (all folds): {roc_auc_score(y_full, oof):.6f}")

pd.DataFrame({"id": train_ids, "pred_adaboost_full_data": oof}).to_csv("oof_adaboost_full_data.csv", index=False)
print("Saved: oof_adaboost_full_data.csv")

## Weighted Ensemble


In [43]:
train = pd.read_csv("data/train_combined.csv")
train = treat_dataset(train)

y = train["heart_disease"].astype(int).values

oof_cat = pd.read_csv("submissions/oof_cat_v4_4seed_full_data.csv")
oof_xgb = pd.read_csv("submissions/oof_xgb_shallow_4seed_full_data.csv")

# Use row position instead of merging on ID (since all have same length and order)
p_cat = oof_cat["pred_cat_v4"].values
p_xgb = oof_xgb["pred_xgb_shallow_full_data"].values

# ----------------------------
# 1) Individual AUCs (sanity)
# ----------------------------
auc_cat = roc_auc_score(y, p_cat)
auc_xgb = roc_auc_score(y, p_xgb)
print(f"Cat OOF AUC: {auc_cat:.6f}")
print(f"XGB OOF AUC: {auc_xgb:.6f}")

# ----------------------------
# 2) Weight scan (AUC on blended OOF)
# ----------------------------
best = (-1.0, None)

for w_cat in np.linspace(0, 1, 11):  # 0.0, 0.1, ..., 1.0
    p_blend = w_cat * p_cat + (1 - w_cat) * p_xgb
    auc = roc_auc_score(y, p_blend)
    print(f"Blend w_cat={w_cat:.1f} | OOF AUC={auc:.6f}")
    if auc > best[0]:
        best = (auc, w_cat)

print(f"Best blend: w_cat={best[1]:.2f} | OOF AUC={best[0]:.6f}")

Cat OOF AUC: 0.955533
XGB OOF AUC: 0.955519
Blend w_cat=0.0 | OOF AUC=0.955519
Blend w_cat=0.1 | OOF AUC=0.955538
Blend w_cat=0.2 | OOF AUC=0.955552
Blend w_cat=0.3 | OOF AUC=0.955563
Blend w_cat=0.4 | OOF AUC=0.955569
Blend w_cat=0.5 | OOF AUC=0.955572
Blend w_cat=0.6 | OOF AUC=0.955572
Blend w_cat=0.7 | OOF AUC=0.955567
Blend w_cat=0.8 | OOF AUC=0.955559
Blend w_cat=0.9 | OOF AUC=0.955548
Blend w_cat=1.0 | OOF AUC=0.955533
Best blend: w_cat=0.50 | OOF AUC=0.955572


In [46]:
# create blended test predictions using the best weight
p_xgb = pd.read_csv("submissions/submission_xgb_shallow_4seed_full_data.csv").sort_values("id").reset_index(drop=True)["heart_disease"].values

p_cat = pd.read_csv("submissions/submission_catboost_v4_seed_ens_full_data.csv").sort_values("id").reset_index(drop=True)["heart_disease"].values

p_test_blend = 0.5 * p_cat + (1 - 0.5) * p_xgb

test_ids = pd.read_csv("data/test.csv")["id"].values

submission_blend = pd.DataFrame({"id": test_ids, "heart_disease": p_test_blend})
submission_blend.to_csv("submission_cat_xgb_blend_w_50_full_data.csv", index=False)

print("Saved: submission_cat_xgb_blend_w_50_full_data.csv")

Saved: submission_cat_xgb_blend_w_50_full_data.csv


## Hill Climb

cat OOF AUC: 0.955533
xgb OOF AUC: 0.955519
cat_te OOF AUC: 0.955468
xgb_te OOF AUC: 0.955483
realmlp OOF AUC: 0.953321
Start: cat | AUC=0.955533
Step 1: add xgb w=0.5 | AUC=0.955572
Step 2: add xgb_te w=0.15 | AUC=0.955576
Step 3: add cat_te w=0.1 | AUC=0.955577
Step 4: add xgb_te w=0.05 | AUC=0.955577

Hillclimb final AUC: 0.9555773109573841
Recipe: cat -> xgb@0.5 -> xgb_te@0.15 -> cat_te@0.1 -> xgb_te@0.05

Meta-logreg (meta-OOF) AUC: 0.9555541253239047

In [9]:
# 0) Load labels (all data)
train = pd.read_csv("data/train_combined.csv")
train = treat_dataset(train)

y = train["heart_disease"].astype(int).values

# 1) OOF specs (keep yours)
oof_specs = [
    ("cat",    "submissions/oof_cat_v4_4seed_full_data.csv",      "pred_cat_v4"),
    ("xgb",    "submissions/oof_xgb_shallow_4seed_full_data.csv",       "pred_xgb_shallow_full_data"),
    #("cat_te", "submissions/oof_cat_te_full_data.csv",            "pred_cat_te_full_data"),
    #("xgb_te", "submissions/oof_xgb_shallow_te_full_data.csv",    "pred_xgb_shallow_te_full_data"),
    #("realmlp","submissions/oof_realmlp_full_data.csv",           "pred_realmlp_full_data"),
]

# Load all OOF predictions WITHOUT merging (use row order instead)
P_dict = {}
for name, path, col in oof_specs:
    tmp = pd.read_csv(path)
    print(f"{name}: rows={len(tmp)}")
    if col not in tmp.columns:
        raise KeyError(f"{name}: column '{col}' not found. Available: {tmp.columns.tolist()}")
    P_dict[name] = tmp[col].values

# Stack into matrix
model_names = [s[0] for s in oof_specs]
P = np.column_stack([P_dict[name] for name in model_names])

print(f"\nP shape: {P.shape}")
print(f"y shape: {y.shape}")
assert len(y) == P.shape[0], f"Mismatch: y has {len(y)} rows, P has {P.shape[0]}"

# Base AUCs
for j, name in enumerate(model_names):
    auc = roc_auc_score(y, P[:, j])
    print(f"{name} OOF AUC: {auc:.6f}")

cat: rows=630270
xgb: rows=630270

P shape: (630270, 2)
y shape: (630270,)
cat OOF AUC: 0.955533
xgb OOF AUC: 0.955519


In [10]:
# Base AUCs
for j, name in enumerate(model_names):
    print(f"{name} OOF AUC: {roc_auc_score(y, P[:, j]):.6f}")

# ----------------------------
# 2) Hill-climb ensemble (greedy, AUC-optimized)
# ----------------------------
def hillclimb_auc(P, y, names, weight_grid=None, max_iters=50):
    """
    Greedy AUC hillclimb:
    - Start with best single model
    - Repeatedly add one model with best AUC gain using weights in weight_grid
    """
    if weight_grid is None:
        weight_grid = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

    # start: best single
    aucs = [roc_auc_score(y, P[:, j]) for j in range(P.shape[1])]
    best_j = int(np.argmax(aucs))
    ens = P[:, best_j].copy()
    chosen = [names[best_j]]
    best_auc = aucs[best_j]

    print(f"Start: {chosen[-1]} | AUC={best_auc:.6f}")

    improved = True
    it = 0
    while improved and it < max_iters:
        improved = False
        it += 1
        best_try = (best_auc, None, None)  # (auc, model_name, w)

        for j, name in enumerate(names):
            cand = P[:, j]
            for w in weight_grid:
                # new = (1-w)*ens + w*cand
                pred = (1 - w) * ens + w * cand
                auc = roc_auc_score(y, pred)
                if auc > best_try[0] + 1e-8:
                    best_try = (auc, name, w)

        if best_try[1] is not None:
            best_auc, name, w = best_try
            ens = (1 - w) * ens + w * P[:, names.index(name)]
            chosen.append(f"{name}@{w}")
            print(f"Step {it}: add {name} w={w} | AUC={best_auc:.6f}")
            improved = True

    return ens, best_auc, chosen

ens_hc, auc_hc, recipe = hillclimb_auc(P, y, model_names)
print("\nHillclimb final AUC:", auc_hc)
print("Recipe:", " -> ".join(recipe))

# ----------------------------
# 3) Meta logistic regression (with meta-OOF for honest eval)
# ----------------------------
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_oof = np.zeros(len(y), dtype=float)

for tr_idx, va_idx in skf.split(P, y):
    meta = LogisticRegression(max_iter=4000, solver="lbfgs")
    meta.fit(P[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta.predict_proba(P[va_idx])[:, 1]

auc_meta = roc_auc_score(y, meta_oof)
print("\nMeta-logreg (meta-OOF) AUC:", auc_meta)

cat OOF AUC: 0.955533
xgb OOF AUC: 0.955519
Start: cat | AUC=0.955533
Step 1: add xgb w=0.5 | AUC=0.955572
Step 2: add cat w=0.05 | AUC=0.955572

Hillclimb final AUC: 0.9555723970770835
Recipe: cat -> xgb@0.5 -> cat@0.05

Meta-logreg (meta-OOF) AUC: 0.9555663592154178


In [12]:
# Kaggle based on cat -> xgb@0.5 -> xgb_te@0.15 -> cat_te@0.1 -> xgb_te@0.05
import pandas as pd

# ---- weights from hillclimb recipe ----
W_CAT    = 0.363375
W_XGB    = 0.363375
W_XGB_TE = 0.17825
W_CAT_TE = 0.095

# ---- load submissions (rename pred column for safety) ----
cat    = pd.read_csv("submissions/submission_catboost_v4_seed_ens_full_data.csv").rename(columns={"heart_disease": "p_cat"})
xgb    = pd.read_csv("submissions/submission_xgb_shallow_4seed_full_data.csv").rename(columns={"heart_disease": "p_xgb"})
xgb_te = pd.read_csv("submission_xgb_shallow_te_full_data.csv").rename(columns={"heart_disease": "p_xgb_te"})
cat_te = pd.read_csv("submission_catboost_te_full_data.csv").rename(columns={"heart_disease": "p_cat_te"})

# ---- merge by id (fail fast if mismatch) ----
df = cat.merge(xgb, on="id", how="inner") \
        .merge(xgb_te, on="id", how="inner") \
        .merge(cat_te, on="id", how="inner")

print("Merged test rows:", len(df))

# ---- weighted blend ----
df["heart_disease"] = (
    W_CAT    * df["p_cat"] +
    W_XGB    * df["p_xgb"] +
    W_XGB_TE * df["p_xgb_te"] +
    W_CAT_TE * df["p_cat_te"]
)

out_path = "submission_hillclimb_blend_full_data.csv"
df[["id", "heart_disease"]].to_csv(out_path, index=False)
print("Saved:", out_path)

Merged test rows: 270000
Saved: submission_hillclimb_blend_full_data.csv


## Ridge

Ridge alpha=0.0001 | meta-OOF AUC=0.955577
Ridge alpha=0.001  | meta-OOF AUC=0.955577
Ridge alpha=0.01   | meta-OOF AUC=0.955577
Ridge alpha=0.1    | meta-OOF AUC=0.955577
Ridge alpha=1.0    | meta-OOF AUC=0.955577
Ridge alpha=10.0   | meta-OOF AUC=0.955577
Ridge alpha=100.0  | meta-OOF AUC=0.955574
Best Ridge: alpha=10.0 | meta-OOF AUC=0.955577

In [11]:
from sklearn.linear_model import Ridge

# ----------------------------
# Ridge meta-model (meta-OOF)
# ----------------------------
alphas = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best = (-1.0, None)

for a in alphas:
    meta_oof = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in skf.split(P, y):
        # Ridge outputs unbounded scores; that's fine for AUC (ranking)
        meta = Ridge(alpha=a, random_state=42)
        meta.fit(P[tr_idx], y[tr_idx])
        meta_oof[va_idx] = meta.predict(P[va_idx])

    auc = roc_auc_score(y, meta_oof)
    print(f"Ridge alpha={a:<6} | meta-OOF AUC={auc:.6f}")

    if auc > best[0]:
        best = (auc, a)

print(f"Best Ridge: alpha={best[1]} | meta-OOF AUC={best[0]:.6f}")

Ridge alpha=0.0001 | meta-OOF AUC=0.955570
Ridge alpha=0.001  | meta-OOF AUC=0.955570
Ridge alpha=0.01   | meta-OOF AUC=0.955570
Ridge alpha=0.1    | meta-OOF AUC=0.955570
Ridge alpha=1.0    | meta-OOF AUC=0.955570
Ridge alpha=10.0   | meta-OOF AUC=0.955570
Ridge alpha=100.0  | meta-OOF AUC=0.955571
Best Ridge: alpha=100.0 | meta-OOF AUC=0.955571


In [ ]:
# ridge submission using the best alpha

# ---- load OOFs (must be aligned to train_combined ids) ----
train = pd.read_csv("data/train_combined.csv")
train = treat_dataset(train)

oof_cat    = pd.read_csv("submissions/oof_cat_v4_4seed_full_data.csv").rename(columns={"pred_cat_v4_full_data": "cat"})
oof_xgb    = pd.read_csv("submissions/oof_xgb_shallow_full_data.csv").rename(columns={"pred_xgb_shallow_full_data": "xgb"})
oof_xgb_te = pd.read_csv("submissions/oof_xgb_shallow_te_full_data.csv").rename(columns={"pred_xgb_shallow_te_full_data": "xgb_te"})
oof_cat_te = pd.read_csv("submissions/oof_cat_te_full_data.csv").rename(columns={"pred_cat_te_full_data": "cat_te"})

df_oof = train[["id", "heart_disease"]].merge(oof_cat, on="id", how="inner") \
                                      .merge(oof_xgb, on="id", how="inner") \
                                      .merge(oof_xgb_te, on="id", how="inner") \
                                      .merge(oof_cat_te, on="id", how="inner")

X_oof = df_oof[["cat", "xgb", "xgb_te", "cat_te"]].values
y_oof = df_oof["heart_disease"].astype(int).values

# ---- fit Ridge (alpha=10.0 from your sweep) ----
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_oof, y_oof)

print("Ridge weights:", ridge.coef_, "intercept:", ridge.intercept_)

# ---- load test submissions to build X_test in same column order ----
cat    = pd.read_csv("submission_catboost_v4_seed_ens_full_data.csv").rename(columns={"heart_disease": "cat"})
xgb    = pd.read_csv("submission_xgb_shallow_4seed_full_data.csv").rename(columns={"heart_disease": "xgb"})
xgb_te = pd.read_csv("submission_xgb_shallow_te_full_data.csv").rename(columns={"heart_disease": "xgb_te"})
cat_te = pd.read_csv("submission_catboost_te_full_data.csv").rename(columns={"heart_disease": "cat_te"})

df_test = cat.merge(xgb, on="id", how="inner") \
             .merge(xgb_te, on="id", how="inner") \
             .merge(cat_te, on="id", how="inner")

X_test = df_test[["cat", "xgb", "xgb_te", "cat_te"]].values

# Ridge outputs unbounded scores; that's fine for AUC.
# Kaggle usually accepts any continuous score, but if you want to keep [0,1], squash:
scores = ridge.predict(X_test)

# Optional: min-max to [0,1] (does not change ranking within the test set)
scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

out = pd.DataFrame({"id": df_test["id"], "heart_disease": scores})
out_path = "submission_ridge_stack_full_data.csv"
out.to_csv(out_path, index=False)
print("Saved:", out_path)

# Notes

- Guide on what they did: https://www.kaggle.com/competitions/playground-series-s6e2/discussion/674420

# Try

- RealMLP trained on identical folds (leveraging internal embeddings) | this notebook: https://www.kaggle.com/code/masayakawamata/trust-your-cv-a-robust-ensemble-strategy?scriptVersionId=298118210 

- Try TargetEncoding with XGBoost (https://www.kaggle.com/code/masayakawamata/trust-your-cv-a-robust-ensemble-strategy?scriptVersionId=298118210)

- Bayes error

- p = 1 / (1 + exp(-logit(p)))

- Ensemble: Hill Climbing

- Ensemble: Linear Regression

- Apply KKN (with K=4);

- Hyperparameter search again

# Results

- Cat OOF AUC all data: 0.955533
- Cat OOF AUC  “original data”: 0.9554778464282366
- Cat TE all data: OOF AUC (all folds): 0.955468
* XGB OOF AUC all data: 0.955519
* XGB OOF AUC original: 0.9553810007540939
- XGB TE all data: OOF AUC (all folds): 0.955483
- RealMLP all data:  OOF AUC: 0.953321
* RealMLP OOF AUC “original data”: 0.953433


Ensembles
Best blend: w_cat=0.50 | OOF AUC=0.955572